# Experimental Finder API

This notebook introduces the developer-friendly experimental API in `TCT.experimental`. The functions wrap common Translator query boilerplate so you can start with labels like `"asthma"` or CURIEs like `"MONDO:0004979"`.

The API is experimental: import from `TCT.experimental`, pin behavior in your own notebooks or applications, and expect the interface to evolve as it graduates into the stable package surface.

Use this notebook when you want a concise pathfinder or neighborhood-finder workflow. If you need fine-grained endpoint selection, custom TRAPI query construction, manual parser workflows, or visualization setup, see the legacy notebooks linked from the README.

In [ ]:
from TCT.experimental import (
    clear_translator_resource_cache,
    get_translator_resources,
    neighborhood_finder,
    pathfinder,
)

## What the wrapper handles

The experimental API combines several lower-level TCT steps:

- Resolve names with Name Resolver when inputs are labels.
- Normalize CURIEs and labels with Node Normalizer.
- Load and cache Translator API metadata.
- Select compatible predicates and APIs from the MetaKG.
- Query selected Translator KPs in parallel.
- Return a `FinderResult` object with parsed graph sections surfaced directly.

Live result counts vary with Translator metadata and KP availability, so examples below focus on reusable code patterns and result shape.

## Resource Cache

Translator API metadata is expensive to fetch. The experimental API loads it lazily on the first query and reuses it for later calls in the same Python process.

Use `get_translator_resources()` when you want to preload metadata once, pass it to multiple calls, or make caching explicit in a notebook. Use `clear_translator_resource_cache()` when you want the next call to refetch metadata.

In [ ]:
resources = get_translator_resources()

{
    "api_count": len(resources.api_names),
    "meta_kg_shape": resources.meta_kg.shape,
    "predicate_api_count": len(resources.api_predicates),
}

## Pathfinder Quick Start

`pathfinder(start, end, intermediate_categories)` searches for two-hop paths between two concepts.

Inputs:

- `start`: start concept as a display string or CURIE.
- `end`: end concept as a display string or CURIE.
- `intermediate_categories`: allowed categories for the connecting node. Short names such as `"Gene"` are automatically converted to `"biolink:Gene"`.

The return value is a `FinderResult` with `.knowledge_graph`, `.results`, `.auxiliary_graphs`, `.resolved_nodes`, `.raw`, and `.to_dict()`.

In [ ]:
paths = pathfinder(
    start="asthma",
    end="albuterol",
    intermediate_categories=["Gene", "Protein"],
    resources=resources,
)
paths.resolved_nodes

In [ ]:
{
    "node_count": len(paths.knowledge_graph.get("nodes", {})),
    "edge_count": len(paths.knowledge_graph.get("edges", {})),
    "result_count": len(paths.results),
    "raw_sections": list(paths.to_dict().keys()),
}

## Neighborhood Finder Quick Start

`neighborhood_finder(node, neighbor_categories)` searches for one-hop neighbors of a concept. `node` can be a single string/CURIE or a list of strings/CURIEs.

The optional `node_categories` argument lets you provide the source category directly when you know it. Short category names like `"Disease"` are converted to Biolink categories.

In [ ]:
neighbors = neighborhood_finder(
    node="MONDO:0004979",
    neighbor_categories=["SmallMolecule", "Drug"],
    node_categories=["Disease"],
    resources=resources,
)
neighbors.resolved_nodes

In [ ]:
{
    "node_count": len(neighbors.knowledge_graph.get("nodes", {})),
    "edge_count": len(neighbors.knowledge_graph.get("edges", {})),
    "result_count": len(neighbors.results),
    "raw_sections": list(neighbors.to_dict().keys()),
}

## Multiple Input Neighborhood Queries

Pass a list of labels or CURIEs to query neighborhoods for multiple source nodes with the same neighbor categories.

In [ ]:
multi_neighbors = neighborhood_finder(
    node=["asthma", "MONDO:0004979"],
    neighbor_categories=["Gene"],
    resources=resources,
)
multi_neighbors.resolved_nodes

## CURIE Inputs

When an input contains `:` and no spaces, the experimental API treats it as a CURIE and skips Name Resolver. It still uses Node Normalizer to get the preferred identifier, label, and categories. This makes CURIE examples more reproducible when Name Resolver rankings change.

In [ ]:
curie_neighbors = neighborhood_finder(
    node="MONDO:0004979",
    neighbor_categories=["Gene"],
    node_categories=["Disease"],
    resources=resources,
)
curie_neighbors.resolved_nodes["node"]

## Advanced Controls

The wrapper still exposes useful controls for common tuning:

- `start_categories`, `end_categories`, and `node_categories` override inferred categories.
- `predicates_subset` narrows neighborhood predicates after MetaKG selection.
- `attribute_constraints` passes TRAPI attribute constraints into the query edge.
- `resources` lets you reuse metadata or pass a filtered `TranslatorResources` object.

For lower-level control over endpoint lists, predicate dictionaries, query JSON, and result parsing, use the detailed PathFinder, NeighborhoodFinder, NetworkFinder, KG overview, and visualization notebooks.

In [ ]:
constrained_neighbors = neighborhood_finder(
    node="asthma",
    neighbor_categories=["SmallMolecule"],
    node_categories=["Disease"],
    predicates_subset=["biolink:treats", "biolink:ameliorates"],
    attribute_constraints=[
        {
            "id": "biolink:knowledge_level",
            "operator": "==",
            "value": "knowledge_assertion",
        }
    ],
    resources=resources,
)

## Refreshing Metadata

Use `refresh=True` to refetch metadata, or clear the cache before the next query.

In [ ]:
fresh_resources = get_translator_resources(refresh=True)
clear_translator_resource_cache()